#Preparación de Setup

In [0]:
print("Hola data brincks, ya estoy conectado")

Hola data brincks, ya estoy conectado


In [0]:
%sh
wget -q "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx" -O /tmp/retail.xlsx
echo "Descarga completada"

Descarga completada


In [0]:
%pip install openpyxl


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

#Extracción

In [0]:
import pandas as pd
from pyspark.sql import functions as F

# Leer el Excel con pandas
pdf = pd.read_excel("/tmp/retail.xlsx", dtype=str)

# Convertir a DataFrame de Spark
df = spark.createDataFrame(pdf)

# Ver cuántos registros hay
print(f"Total de registros: {df.count():,}")

# Ver las primeras filas y el esquema
# En lugar de df.show(5, truncate=False), tirá esto:
display(df.limit(5))

Total de registros: 541,909


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


#Transformación
##Limpiar Datos

In [0]:
df_limpio = df \
  .dropna(subset=["CustomerID", "InvoiceNo"]) \
  .filter(~F.col("InvoiceNo").startswith("C")) \
  .withColumn("Quantity", F.col("Quantity").cast("integer")) \
  .withColumn("UnitPrice", F.col("UnitPrice").cast("double")) \
  .filter(F.col("Quantity") > 0) \
  .filter(F.col("UnitPrice") > 0)

print(f"Registros después de limpiar: {df_limpio.count():,}")

Registros después de limpiar: 397,884


#Transformación
##Calcular métricas de negocio reales

In [0]:
df_ventas = df_limpio.withColumn(
    "total_venta",
    F.col("Quantity") * F.col("UnitPrice")
)

ventas_por_pais = df_ventas \
    .groupBy("Country") \
    .agg(
        F.round(F.sum("total_venta"), 2).alias("revenue_total"),
        F.countDistinct("InvoiceNo").alias("cant_facturas"),
        F.countDistinct("CustomerID").alias("cant_clientes")
    ) \
    .orderBy(F.col("revenue_total").desc())

top_productos = df_ventas \
    .groupBy("StockCode", "Description") \
    .agg(
        F.sum("Quantity").alias("unidades_vendidas"),
        F.round(F.sum("total_venta"), 2).alias("revenue")
    ) \
    .orderBy(F.col("revenue").desc()) \
    .limit(20)

print("=== TOP 5 PAÍSES ===")
ventas_por_pais.show(5)
print("=== TOP 5 PRODUCTOS ===")
top_productos.show(5)

=== TOP 5 PAÍSES ===
+--------------+-------------+-------------+-------------+
|       Country|revenue_total|cant_facturas|cant_clientes|
+--------------+-------------+-------------+-------------+
|United Kingdom|   7308391.55|        16646|         3920|
|   Netherlands|    285446.34|           94|            9|
|          EIRE|     265545.9|          260|            3|
|       Germany|    228867.14|          457|           94|
|        France|    209024.05|          389|           87|
+--------------+-------------+-------------+-------------+
only showing top 5 rows
=== TOP 5 PRODUCTOS ===
+---------+--------------------+-----------------+---------+
|StockCode|         Description|unidades_vendidas|  revenue|
+---------+--------------------+-----------------+---------+
|    23843|PAPER CRAFT , LIT...|            80995| 168469.6|
|    22423|REGENCY CAKESTAND...|            12402|142592.95|
|   85123A|WHITE HANGING HEA...|            36725|100448.15|
|   85099B|JUMBO BAG RED RET...|  

#Carga

In [0]:
ventas_por_pais.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_ventas_por_pais")

top_productos.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_top_productos")

print("Tablas guardadas exitosamente")

Tablas guardadas exitosamente
